In [1]:
!pip install --q transformers
!pip install --q evaluate
!pip install --q accelerate
!pip install --q transformers[torch]

In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import random
import gensim
import gensim.downloader
from gensim.models import KeyedVectors
import matplotlib.pyplot as plt
from tqdm import tqdm
from tabulate import tabulate

#pytorch
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim import AdamW

#tf model
import tensorflow_hub as hub

#misc
import datasets
import evaluate
from transformers import TrainingArguments, Trainer

#transformers
from transformers import BertForSequenceClassification, AutoModelForSeq2SeqLM, AutoTokenizer, AutoConfig, BertModel, XLNetForSequenceClassification,YosoForSequenceClassification


#sklearn
from sklearn.metrics import classification_report, confusion_matrix

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

In [3]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

**Data Preparation**

In [5]:
!wget -O "amazon.zip" "https://drive.google.com/uc?export=download&id=1MO1EYtIeCsJliaeCxrkqrIWKsowsd3X8&confirm=t&uuid=70bad3da-836c-43dd-90bc-c21da0dfb131&at=AB6BwCAx5FEYhB_nNxz511oJwkhT:1692183100774"

--2023-08-24 14:37:31--  https://drive.google.com/uc?export=download&id=1MO1EYtIeCsJliaeCxrkqrIWKsowsd3X8&confirm=t&uuid=70bad3da-836c-43dd-90bc-c21da0dfb131&at=AB6BwCAx5FEYhB_nNxz511oJwkhT:1692183100774
Resolving drive.google.com (drive.google.com)... 172.217.24.46, 2404:6800:4006:804::200e
Connecting to drive.google.com (drive.google.com)|172.217.24.46|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://doc-14-0g-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/69b8oulpc3j4ujh7q672el5e4fm09ubk/1692887850000/16787182300288898320/*/1MO1EYtIeCsJliaeCxrkqrIWKsowsd3X8?e=download&uuid=70bad3da-836c-43dd-90bc-c21da0dfb131 [following]
--2023-08-24 14:37:32--  https://doc-14-0g-docs.googleusercontent.com/docs/securesc/ha0ro937gcuc7l7deffksulhg5h7mbp1/69b8oulpc3j4ujh7q672el5e4fm09ubk/1692887850000/16787182300288898320/*/1MO1EYtIeCsJliaeCxrkqrIWKsowsd3X8?e=download&uuid=70bad3da-836c-43dd-90bc-c21da0dfb131
Resolving doc-14-0g-docs.

In [6]:
!unzip "amazon.zip"

Archive:  amazon.zip
  inflating: test.csv                
  inflating: train.csv               
  inflating: validation.csv          
  inflating: amazon_translated_body_and_title_with_originals.csv  
  inflating: amazon_translated_body_and_title_with_originals_all_stars.csv  


In [7]:
SEED = 111

# Set the random seed for Python to SEED
random.seed(SEED)

# Set the random seed for numpy to SEED
np.random.seed(SEED)

# Set the random seed for torch to SEED
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [8]:
def prepareData(sentences, labels, tokenizer, max_length=256, batch_size=32):
    encoded_inputs = tokenizer(list(sentences), padding='max_length', truncation=True, max_length=max_length, return_tensors='pt')
    input_ids = encoded_inputs['input_ids']
    attention_mask = encoded_inputs['attention_mask']
    dataset = torch.utils.data.TensorDataset(input_ids, attention_mask, torch.tensor(labels))
    dataLoader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataLoader

In [9]:
def prepareDataTruncation(sentences, labels, tokenizer, N, M, max_length=256, batch_size=32):
    tokenized_inputs = [tokenizer.tokenize(sentences[i]) for i in range(len(list(sentences)))]
    for i in range(len(tokenized_inputs)):
        if len(tokenized_inputs[i]) > max_length:
            tokenized_inputs[i] = tokenized_inputs[i][:N] + tokenized_inputs[i][-M:]
        tokenized_inputs[i] = tokenizer.convert_tokens_to_string(tokenized_inputs[i])
    return prepareData(tokenized_inputs, labels, tokenizer, max_length=max_length, batch_size=batch_size)

**IMDB Dataset Training**

**Dataset Evaluation**

In [10]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [11]:
def tokenize_function(examples):
    tokenized_inputs = [tokenizer.tokenize(examples[reviewType][i]) for i in range(len(list(examples[reviewType])))]
    for i in range(len(tokenized_inputs)):
        if len(tokenized_inputs[i]) > N+M:
            tokenized_inputs[i] = tokenized_inputs[i][:N] + tokenized_inputs[i][-M:]
        tokenized_inputs[i] = tokenizer.convert_tokens_to_string(tokenized_inputs[i])
    tokenized_inputs = tokenizer(tokenized_inputs,padding="max_length", truncation=True, max_length=N+M)
    return tokenized_inputs

In [12]:
def print_using_tabulate(data):
    table_data = []
    for key, values in data.items():
        if key != 'macro avg' and key != 'weighted avg':
            if isinstance(values, dict):
                row = [key, values['precision'], values['recall'], values['f1-score'], values['support']]
                table_data.append(row)

    # Print the classification report using tabulate
    headers = ['Class', 'Precision', 'Recall', 'F1-Score', 'Support']
    m_table = tabulate(table_data, headers=headers, tablefmt='psql',floatfmt=".4f")
    print(m_table)

In [13]:
def evaluateModel(model, test_data, target_names):
    classificationReports = []
    model.eval()
    np_predictions = []
    np_y_eval = []
    for batch in test_data:
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        predictions = model(input_ids=input_ids, attention_mask=attention_mask)
        _, outclass_predictions = torch.max(predictions.logits, 1)
        np_predictions.extend(outclass_predictions.cpu().numpy())
        np_y_eval.extend(labels.cpu().numpy())
    np_predictions = np.array(np_predictions)
    np_y_eval = np.array(np_y_eval)
    classificationReports = classification_report(np_y_eval, np_predictions, target_names=target_names, output_dict=True)
    # Confusion matrix for each label
    return classificationReports

In [14]:
def train(model, training_args, dataset, numOfExamples, imdbOrAmazon="imdb"):
    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    small_train_dataset = tokenized_datasets["train"].shuffle(seed=SEED).select(range(numOfExamples))
    if imdbOrAmazon == "imdb":
        trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=small_train_dataset,
        eval_dataset=tokenized_datasets['test'],
        compute_metrics=compute_metrics,
        )
    else:
        trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=small_train_dataset,
        eval_dataset=tokenized_datasets['validation'],
        compute_metrics=compute_metrics,
        )
    trainer.train()

In [15]:
def evaluateTranslatedTestSetAndRegularTestSet(model, tokenizer, dataset, translatedDataset, targetNames, addedText, batch_size=8):
    tranlsatedTestSetTrunc = prepareDataTruncation(translatedDataset['translated_body'], translatedDataset['labels'], tokenizer, N, M, max_length=N+M, batch_size=batch_size)
    classificationReports = evaluateModel(model, tranlsatedTestSetTrunc, targetNames)
    print(f"\033[1m{addedText}\033[0m")
    print("Classification Report Translated Test Set")
    print_using_tabulate(classificationReports)
    testSet = prepareDataTruncation(dataset['test'][reviewType], dataset['test']['labels'], tokenizer, N, M, max_length=N+M, batch_size=batch_size)
    print("Classification Report Test Set")
    classificationReports = evaluateModel(model, testSet, targetNames)
    print_using_tabulate(classificationReports)

**IMDB Dataset Start & End Only Trial**

Logically, when looking at reviews, most likely the sentiment will be at the start or the end (or both) of the review and the more in-depth details of the product/movie/etc will be in the middle.
Ideally, we would have liked to process all the review, mark N start tokens with "start_token ... \start_token" and mark M end tokens with "end_token ... \end_token".
Then, train bert while pooling out these indices from the last hidden layer. Then, through K fully connected layers, normalization, dropout, etc ...  and finally a classifier.
This operation will be too expensive, practically infeasible in our case due to limited resources and very long reviews.
Therefore, what we will do instead is preprocess the dataset such that the first N tokens are concatenated to the last M tokens to form a new review with the same sentiment as before.

In [16]:
saveStrategy = "no"
targetNames = ['negative', 'positive']

In [17]:
translatedTestSet = pd.read_csv('amazon_translated_body_and_title_with_originals.csv')
translatedTestSet.head(5)

,Unnamed: 0,stars,review_body,review_title,language,translated_title,translated_body
0,0,0,"Leider, leider nach einmal waschen ausgebliche...",Leider nicht zu empfehlen,de,Unfortunately not recommended,"Unfortunately, unfortunately faded after one w..."
1,1,0,zunächst macht der Anker Halter einen soliden ...,Gummierung nach 6 Monaten kaputt,de,Rubber broken after 6 months,"first of all, the anchor holder makes a solid ..."
2,2,0,Siegel sowie Verpackung war beschädigt und war...,Flohmarkt ware,de,flea market goods,Seal and packaging was damaged and item was us...
3,3,0,Habe dieses Produkt NIE erhalten und das Geld ...,Katastrophe,de,catastrophe,NEVER received this product and the money was ...
4,4,0,Die Träger sind schnell abgerissen,Reißverschluss klemmt,de,Zipper is stuck,The straps ripped off quickly


In [18]:
translatedTestSet.drop(columns=['Unnamed: 0'], axis=0, inplace=True)
translatedTestSet.rename(columns={'stars':'labels'}, inplace=True)
translatedTestSet.head(5)

,labels,review_body,review_title,language,translated_title,translated_body
0,0,"Leider, leider nach einmal waschen ausgebliche...",Leider nicht zu empfehlen,de,Unfortunately not recommended,"Unfortunately, unfortunately faded after one w..."
1,0,zunächst macht der Anker Halter einen soliden ...,Gummierung nach 6 Monaten kaputt,de,Rubber broken after 6 months,"first of all, the anchor holder makes a solid ..."
2,0,Siegel sowie Verpackung war beschädigt und war...,Flohmarkt ware,de,flea market goods,Seal and packaging was damaged and item was us...
3,0,Habe dieses Produkt NIE erhalten und das Geld ...,Katastrophe,de,catastrophe,NEVER received this product and the money was ...
4,0,Die Träger sind schnell abgerissen,Reißverschluss klemmt,de,Zipper is stuck,The straps ripped off quickly


**BERT CASED**

****N = 128, M = 382; Overall 510 tokens****

In [19]:
N = 128
M = 382
reviewType = "text"

In [20]:
from datasets import load_dataset
imdbDataset = load_dataset('imdb')

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset imdb downloaded and prepared to /root/.cache/huggingface/datasets/imdb/plain_text/1.0.0/2fdd8b9bcadd6e7055e742a706876ba43f19faee861df134affd7a3f60fc38a1. Subsequent calls will reuse this data.


  0%|          | 0/3 [00:00<?, ?it/s]

In [21]:
imdbDataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [22]:
imdbDataset = imdbDataset.rename_column("label", "labels")

In [23]:
metric = evaluate.load("accuracy")
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [24]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [25]:
enLanguageModelStartEndTrial_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initi

In [26]:
train(enLanguageModelStartEndTrial_N_128_M_382, training_args, imdbDataset, imdbDataset['train'].num_rows)

  0%|          | 0/25 [00:00<?, ?ba/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors


  0%|          | 0/25 [00:00<?, ?ba/s]

  0%|          | 0/50 [00:00<?, ?ba/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.297700,0.246911,0.923160


In [27]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_128_M_382, tokenizer, imdbDataset, targetNames, "bert-base-cased-2-labels-510-tokens-head+tail", batch_size=8)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_128_M_382, tok     │
│   2                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
TypeError: evaluateTranslatedTestSetAndRegularTestSet() missing 1 required positional argument: 'addedText'

****repeat with N=M=64; overall 128 tokens****

In [ ]:
N = 64
M = 64
reviewType = "text"

In [ ]:
enLanguageModelStartEndTrial_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_64_M_64, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_64_M_64, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-cased-2-labels-N-64-M-64-head+tail", batch_size=8)

****repeat with N=510, M=0; overall 510 tokens****

In [ ]:
N = 510
M = 0
reviewType = "text"

In [ ]:
enLanguageModelStartEndTrial_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_510_M_0, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_510_M_0, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-cased-2-labels-N-510-M-0-head-only", batch_size=8)

****repeat with N=0, M=510; overall 510 tokens****

In [ ]:
N = 0
M = 510
reviewType = "text"

In [ ]:
enLanguageModelStartEndTrial_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_0_M_510, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_0_M_510, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-cased-2-labels-N-0-M-510-tail-only", batch_size=8)

**BERT UNCASED**

****N = 128, M = 382; Overall 510 tokens****

In [ ]:
N = 128
M = 382
reviewType = "text"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
enLanguageModelStartEndTrial_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_128_M_382, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_128_M_382, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-uncased-2-labels-N-128-M-382-head+tail", batch_size=8)

**N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "text"

In [ ]:
enLanguageModelStartEndTrial_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_64_M_64, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
train(enLanguageModelStartEndTrial_N_64_M_64, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_64_M_64, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-uncased-2-labels-N-64-M-64-head+tail", batch_size=8)

**N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "text"

In [ ]:
enLanguageModelStartEndTrial_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_510_M_0, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_510_M_0, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-uncased-2-labels-N-510-M-0-head-only", batch_size=8)

**N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "text"

In [ ]:
enLanguageModelStartEndTrial_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(enLanguageModelStartEndTrial_N_0_M_510, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(enLanguageModelStartEndTrial_N_0_M_510, tokenizer, imdbDataset, translatedTestSet, targetNames, "bert-base-uncased-2-labels-N-0-M-510-tail-only", batch_size=8)

**XLNET IMDB**

**N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "text"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased")
XLNETModel_N_128_M_382 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_128_M_382, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_128_M_382, tokenizer, imdbDataset, translatedTestSet, targetNames, "xlnet-base-cased-2-labels-N-128-M-382-head+tail", batch_size=8)

**N=64, M=64, overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "text"

In [ ]:
XLNETModel_N_64_M_64 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_64_M_64, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_64_M_64, tokenizer, imdbDataset, translatedTestSet, targetNames, "xlnet-base-cased-2-labels-N-64-M-64-head+tail", batch_size=8)

**N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "text"

In [ ]:
XLNETModel_N_510_M_0 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_510_M_0, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_510_M_0, tokenizer, imdbDataset, translatedTestSet, targetNames, "xlnet-base-cased-2-labels-N-510-M-0-head-only", batch_size=8)

**N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "text"

In [ ]:
XLNETModel_N_0_M_510 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_0_M_510, training_args, imdbDataset, imdbDataset['train'].num_rows)

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_0_M_510, tokenizer, imdbDataset, translatedTestSet, targetNames, "xlnet-base-cased-2-labels-N-0-M-510-tail-only", batch_size=8)

**Amazon En Dataset**

In [ ]:
def dropColumns(df):
    df.drop(columnsToDrop, axis=1, inplace=True)
    return df

In [ ]:
translatedTestSetAllStars = pd.read_csv('amazon_translated_body_and_title_with_originals_all_stars.csv')
translatedTestSetAllStars.head(5)

In [ ]:
translatedTestSetAllStars.drop(columns=['Unnamed: 0'], axis=0, inplace=True)
translatedTestSetAllStars.rename(columns={'stars':'labels'}, inplace=True)

In [ ]:
trainingSetAmazon = pd.read_csv('train.csv')
validationSetAmazon = pd.read_csv('validation.csv')
testSetAmazon = pd.read_csv('test.csv')

In [ ]:
columnsToDrop = ['Unnamed: 0', 'review_id', 'product_id', 'reviewer_id', 'product_category']
trainingSetAmazon = dropColumns(trainingSetAmazon)
validationSetAmazon = dropColumns(validationSetAmazon)
testSetAmazon = dropColumns(testSetAmazon)

In [ ]:
def extractLanguage(language, df):
    new_df = df.copy()
    exclude = np.where(new_df['language'] != language)
    new_df.drop(exclude[0],axis=0, inplace=True)
    return new_df
enOnlyTrain = extractLanguage('en', trainingSetAmazon)
enOnlyDev = extractLanguage('en', validationSetAmazon)
enOnlyTest = extractLanguage('en', testSetAmazon)

In [ ]:
enOnlyTrain.head(5)

In [ ]:
enOnlyTrain = datasets.Dataset.from_pandas(enOnlyTrain)
enOnlyDev = datasets.Dataset.from_pandas(enOnlyDev)
enOnlyTest = datasets.Dataset.from_pandas(enOnlyTest)

In [ ]:
en_only_dataset = datasets.DatasetDict({"train": enOnlyTrain, 'validation': enOnlyDev, 'test': enOnlyTest})
en_only_dataset

In [ ]:
en_only_dataset = en_only_dataset.rename_column('stars', 'labels')

In [ ]:
en_only_dataset = en_only_dataset.remove_columns('__index_level_0__')

In [ ]:
def reduceOne(examples):
    examples['labels'] = examples['labels'] - 1
    return examples

In [ ]:
en_only_dataset = en_only_dataset.map(reduceOne)

In [ ]:
translatedTestSetAllStars['labels'] = translatedTestSetAllStars['labels'] - 1

**Amazon Dataset English Only Model**

**BERT CASED**

**Five labels; N=128, M=382; overall 510 tokens**

In [ ]:
metric = evaluate.load("accuracy")
numLabels = 5
targetNames = ['Negative', 'Somewhat Negative', 'Neutral', 'Somewhat Positive', 'Positive']

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
amazonEnLanguageModel_five_labels_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-4, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_128_M_382, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_128_M_382, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-cased-5-labels-N-128-M-382-head+tail", batch_size=8)

**Five labels; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_five_labels_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_64_M_64, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_64_M_64, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-cased-5-labels-N-64-M-64-head+tail", batch_size=8)

**Five labels; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_five_labels_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_510_M_0, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_510_M_0, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-cased-5-labels-N-510-M-0-head-only", batch_size=8)

**Five labels; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_five_labels_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_0_M_510, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_0_M_510, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-cased-5-labels-N-0-M-510-tail-only", batch_size=8)

**BERT UNCASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", num_labels=numLabels)

**Five labels; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_five_labels_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_128_M_382, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_128_M_382, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-uncased-5-labels-N-128-M-382-head+tail", batch_size=8)

**Five labels; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_five_labels_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_64_M_64, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_64_M_64, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-uncased-5-labels-N-64-M-64-head+tail", batch_size=8)

**Five labels; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_five_labels_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_five_labels_N_510_M_0, training_args, en_only_dataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_five_labels_N_510_M_0, tokenizer, en_only_dataset, translatedTestSetAllStars, targetNames, "bert-base-uncased-5-labels-N-510-M-0-head-only", batch_size=8)

**At this point we got discoureged to keep going as the accuracy is really low**

In [ ]:
def dropColumns(df):
    df.drop(columnsToDrop, axis=1, inplace=True)
    return df

In [ ]:
translatedTestSetAllStars = pd.read_csv('amazon_translated_body_and_title_with_originals_all_stars.csv')
translatedTestSetAllStars.head(5)

In [ ]:
translatedTestSetAllStars.drop(columns=['Unnamed: 0'], axis=0, inplace=True)
translatedTestSetAllStars.rename(columns={'stars':'labels'}, inplace=True)

In [ ]:
trainingSetAmazon = pd.read_csv('train.csv')
validationSetAmazon = pd.read_csv('validation.csv')
testSetAmazon = pd.read_csv('test.csv')

In [ ]:
columnsToDrop = ['Unnamed: 0', 'review_id', 'product_id', 'reviewer_id', 'product_category']
trainingSetAmazon = dropColumns(trainingSetAmazon)
validationSetAmazon = dropColumns(validationSetAmazon)
testSetAmazon = dropColumns(testSetAmazon)

In [ ]:
def extractLanguage(language, df):
    new_df = df.copy()
    exclude = np.where(new_df['language'] != language)
    new_df.drop(exclude[0],axis=0, inplace=True)
    return new_df
enOnlyTrain = extractLanguage('en', trainingSetAmazon)
enOnlyDev = extractLanguage('en', validationSetAmazon)
enOnlyTest = extractLanguage('en', testSetAmazon)

In [ ]:
def changeToBinary(example):
    if example['labels'] >= threshold:
        example['labels'] = 1
    else:
        example['labels'] = 0
    return example

In [ ]:
def removeNeutrals(examples):
    df = pd.DataFrame(examples)
    exclude = np.where(df['labels'] == 3)
    df.drop(exclude[0],axis=0, inplace=True)
    return df

**Train on Amazon, Test on IMDB**

In [ ]:
threshold = 3 # neutrals are included
numLabels = 2
targetNames = ['Negative', 'Positive']
metric = evaluate.load("accuracy")

In [ ]:
updatedDataset = en_only_dataset.map(changeToBinary)

**BERT CASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased', num_labels=numLabels)

**Two labels with Neutral; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-4, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_128_M_382, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_128_M_382, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-N-128-M-382-head+tail", "Amazon", batch_size=8)

**Two labels with Neutral; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_64_M_64, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_64_M_64, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-N-64-M-64-head+tail", "Amazon", batch_size=8)

**Two labels with Neutral; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_510_M_0, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_510_M_0, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-N-510-M-0-head-only", "Amazon", batch_size=8)

**Two labels with Neutral; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_0_M_510, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_0_M_510, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-N-0-M-510-tail-only", "Amazon", batch_size=8)

**BERT UNCASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", num_labels=numLabels)

**Two labels with Neutral; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_128_M_382, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_128_M_382, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-N-128-M-382-head+tail", "Amazon", batch_size=8)

**Two labels with Neutral; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_64_M_64, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_64_M_64, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-N-64-M-64-head+tail", "Amazon", batch_size=8)

**Two labels with Neutral; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_510_M_0, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_510_M_0, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-N-510-M-0-head-only", "Amazon", batch_size=8)

**Two labels with Neutral; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_0_M_510, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_0_M_510, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-N-0-M-510-tail-only", "Amazon", batch_size=8)

**XLNET AMAZON TWO LABELS WITH NEUTRAL**

**Five labels; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased", num_labels=numLabels)
XLNETModel_N_128_M_382 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_128_M_382, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_128_M_382, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-N-128-M-382-head+tail", "Amazon", batch_size=8)

**Two labels with Neutral; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
XLNETModel_N_64_M_64 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_64_M_64, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_64_M_64, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-N-64-M-64-head+tail", "Amazon", batch_size=8)

**Two labels with Neutral; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
XLNETModel_N_510_M_0 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_510_M_0, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_510_M_0, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-N-510-M-0-head-only", "Amazon", batch_size=8)

**Two labels with Neutral; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
XLNETModel_N_0_M_510 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_0_M_510, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_0_M_510, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-N-0-M-510-tail-only", "Amazon", batch_size=8)

**labels 1,2,4,5 only**

In [ ]:
amazonEnLanguageModelNoNeutrals = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

In [ ]:
en_only_dataset = datasets.DatasetDict({"train": enOnlyTrain, 'validation': enOnlyDev, 'test': enOnlyTest})

In [ ]:
en_only_dataset = en_only_dataset.rename_column('stars', 'labels')

In [ ]:
en_only_dataset = en_only_dataset.remove_columns('__index_level_0__')

In [ ]:
en_only_dataset['train'] = datasets.Dataset.from_pandas(removeNeutrals(en_only_dataset['train']))
en_only_dataset['validation'] = datasets.Dataset.from_pandas(removeNeutrals(en_only_dataset['validation']))
en_only_dataset['test'] = datasets.Dataset.from_pandas(removeNeutrals(en_only_dataset['test']))

In [ ]:
updatedDataset = en_only_dataset.map(changeToBinary)

**BERT CASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased', num_labels=numLabels)

**Two labels No Neutral; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-4, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_128_M_382, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_128_M_382, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-No-Neutral-N-128-M-382-head+tail", "Amazon", batch_size=8)

**Two labels No Neutral; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_64_M_64, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_64_M_64, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-No-Neutral-N-64-M-64-head+tail", "Amazon", batch_size=8)

**Two labels No Neutral; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_510_M_0, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_510_M_0, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-No-Neutral-N-510-M-0-head-only", "Amazon", batch_size=8)

**Two labels No Neutral; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_0_M_510, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_0_M_510, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-cased-2-labels-No-Neutral-N-0-M-510-tail-only", "Amazon", batch_size=8)

**BERT UNCASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", num_labels=numLabels)

**Two labels No Neutral; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_128_M_382, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_128_M_382, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-No-Neutral-N-128-M-382-head+tail", "Amazon", batch_size=8)

**Two labels No Neutral; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_64_M_64, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_64_M_64, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-No-Neutral-N-64-M-64-head+tail", "Amazon", batch_size=8)

**Two labels No Neutral; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_510_M_0, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_510_M_0, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-No-Neutral-N-510-M-0-head-only", "Amazon", batch_size=8)

**Two labels No Neutral; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
amazonEnLanguageModel_two_labels_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(amazonEnLanguageModel_two_labels_N_0_M_510, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(amazonEnLanguageModel_two_labels_N_0_M_510, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "bert-base-uncased-2-labels-No-Neutral-N-0-M-510-tail-only", "Amazon", batch_size=8)

**XLNET AMAZON TWO LABELS NO NEUTRAL**

**Two labels No Neutral; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("xlnet-base-cased", num_labels=numLabels)
XLNETModel_N_128_M_382 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_128_M_382, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_128_M_382, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-No-Neutral-N-128-M-382-head+tail", "Amazon", batch_size=8)

**Two labels No Neutral; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
XLNETModel_N_64_M_64 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_64_M_64, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_64_M_64, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-No-Neutral-N-64-M-64-head+tail", "Amazon", batch_size=8)

**Two labels No Neutral; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
XLNETModel_N_510_M_0 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_510_M_0, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_510_M_0, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-No-Neutral-N-510-M-0-head-only", "Amazon", batch_size=8)

**Two labels No Neutral; N=0, M=510; overall 510 tokens**

In [ ]:
N = 256
M = 256
reviewType = "review_body"

In [ ]:
XLNETModel_N_0_M_510 = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(XLNETModel_N_0_M_510, training_args, updatedDataset, 30000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(XLNETModel_N_0_M_510, tokenizer, updatedDataset, imdbDataset['test'], targetNames, "xlnet-base-cased-2-labels-No-Neutral-N-0-M-510-tail-only", "Amazon", batch_size=8)

**MultiLingual Model**

In [ ]:
threshold = 3 # neutrals are included
numLabels = 2
targetNames = ['Negative', 'Positive']

In [ ]:
trainingSetAmazon.head(5)

In [ ]:
def extractMultipleLanguages(langugages, df):
    dfList = []
    for language in languages:
        dfList.append(extractLanguage(language, df))
    newdf = pd.concat(dfList)
    return newdf

In [ ]:
languages = ['en', 'de', 'fr', 'es']
trainingMultiSet = extractMultipleLanguages(languages, trainingSetAmazon)
validationMultiSet = extractMultipleLanguages(languages, validationSetAmazon)
testMultiSet = extractMultipleLanguages(languages, testSetAmazon)

In [ ]:
np.unique(trainingMultiSet['language'])

In [ ]:
multiTrain = datasets.Dataset.from_pandas(trainingMultiSet)
multiValidation = datasets.Dataset.from_pandas(validationMultiSet)
multiTest = datasets.Dataset.from_pandas(testMultiSet)

In [ ]:
multi_dataset = datasets.DatasetDict({"train": multiTrain, 'validation': multiValidation, 'test': multiTest})

In [ ]:
multi_dataset

In [ ]:
multi_dataset = multi_dataset.rename_column('stars', 'labels')
multi_dataset = multi_dataset.remove_columns('__index_level_0__')

In [ ]:
updatedDatasetMulti = multi_dataset.map(changeToBinary)

***BERT MULTILINGUAL***

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
numLabels = 2
targetNames = ['Negative', 'Positive']

**Two labels, With Neutrals; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_128_M_382, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_128_M_382, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-With-Neutrals-N-128-M-382-head+tail", batch_size=8)

**Two labels, With Neutrals; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_64_M_64, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_64_M_64, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-With-Neutrals-N-64-M-64-head+tail", batch_size=8)

**Two labels, With Neutrals; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_510_M_0, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_510_M_0, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-With-Neutrals-N-510-M-0-head-only", batch_size=8)

**Two labels, With Neutrals; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_0_M_510, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_0_M_510, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-With-Neutrals-N-0-M-510-tail-only", batch_size=8)

**BERT MULTILINGUAL UNCASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

**Two labels, With Neutrals; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_128_M_382, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_128_M_382, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-With-Neutrals-N-128-M-382-head+tail", batch_size=8)

**Two labels, With Neutrals; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_64_M_64, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_128_M_382, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-With-Neutrals-N-64-M-64-head+tail", batch_size=8)

**Two labels, With Neutrals; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_510_M_0, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_510_M_0, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-With-Neutrals-N-510-M-0-head-only", batch_size=8)

**Two labels, With Neutrals; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_0_M_510, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_0_M_510, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-With-Neutrals-N-0-M-510-tail-only", batch_size=8)

**NO NEUTRALS**

In [ ]:
multi_dataset['train'] = datasets.Dataset.from_pandas(removeNeutrals(multi_dataset['train']))
multi_dataset['validation'] = datasets.Dataset.from_pandas(removeNeutrals(multi_dataset['validation']))
multi_dataset['test'] = datasets.Dataset.from_pandas(removeNeutrals(multi_dataset['test']))

In [ ]:
updatedDatasetMulti = multi_dataset.map(changeToBinary)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
numLabels = 2
targetNames = ['Negative', 'Positive']

**Two labels, No Neutrals; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=2)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_128_M_382, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_128_M_382, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-No-Neutrals-N-128-M-382-head+tail", batch_size=8)

**Two labels, No Neutrals; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_64_M_64, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_64_M_64, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-No-Neutrals-N-64-M-64-head+tail", batch_size=8)

**Two labels, No Neutrals; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_510_M_0, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_510_M_0, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-No-Neutrals-N-510-M-0-head-only", batch_size=8)

**Two labels, No Neutrals; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
bertMultiLingualModel_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-cased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModel_N_0_M_510, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModel_N_0_M_510, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-cased-2-labels-No-Neutrals-N-0-M-510-tail-only", batch_size=8)

**BERT MULTILINGUAL UNCASED**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

**Two labels, No Neutrals; N=128, M=382; overall 510 tokens**

In [ ]:
N = 128
M = 382
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_128_M_382 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_128_M_382, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_128_M_382, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-No-Neutrals-N-128-M-382-head+tail", batch_size=8)

**Two labels, No Neutrals; N=64, M=64; overall 128 tokens**

In [ ]:
N = 64
M = 64
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_64_M_64 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_64_M_64, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_64_M_64, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-No-Neutrals-N-64-M-64-head+tail", batch_size=8)

**Two labels, No Neutrals; N=510, M=0; overall 510 tokens**

In [ ]:
N = 510
M = 0
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_510_M_0 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_510_M_0, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_510_M_0, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-No-Neutrals-N-510-M-0-head-only", batch_size=8)

**Two labels, No Neutrals; N=0, M=510; overall 510 tokens**

In [ ]:
N = 0
M = 510
reviewType = "review_body"

In [ ]:
bertMultiLingualModelUncased_N_0_M_510 = BertForSequenceClassification.from_pretrained("bert-base-multilingual-uncased", num_labels=numLabels)

In [ ]:
training_args = TrainingArguments(output_dir="test_trainer", optim="adamw_torch", warmup_steps=10000, num_train_epochs=1, weight_decay=1e-1, evaluation_strategy="epoch", save_strategy = saveStrategy)

In [ ]:
train(bertMultiLingualModelUncased_N_0_M_510, training_args, updatedDatasetMulti, 120000, "amazon")

In [ ]:
evaluateTranslatedTestSetAndRegularTestSet(bertMultiLingualModelUncased_N_0_M_510, tokenizer, updatedDatasetMulti, translatedTestSet, targetNames, "bert-base-multilingual-uncased-2-labels-No-Neutrals-N-0-M-510-tail-only", batch_size=8)